# Notebook 02: Classification Heads on Frozen Encoder

Goal of this notebook: Evaluate different classification head architectures on top of the Moirai foundation model. 

We evaluate four distinct architectures:
1. Mean Pooling
2. Single-Scale Attention
3. Single-Scale Multi-Head Attention (MHA)
4. Hierarchical Multi-Head Attention

In [ ]:
import torch
import pandas as pd
from IPython.display import display

from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder
from heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from utils import get_lsst_dataloaders, get_z_loaders, grid_search_heads

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
HEADS_TO_TEST = [8, 16, 32, 64, 128, 384] 

# Hyperparameters
MOIRAI_BATCH_SIZE = 32
HEAD_BATCH_SIZE = 256
LR_GRID = [1e-3]
WD_GRID = [0.05]
MODES = ["shared_context", "independent_context"]

# Results storage
"""
df_mean = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES_TO_TEST)
df_attn = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_mha = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_mha8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_mha16 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_hier = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
"""
df_patch_8_metrics = pd.DataFrame(index=[], columns= ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1',
       'Weighted Precision', 'Weighted Recall', 'Weighted F1'])


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_tr, raw_va, raw_te, num_classes = get_lsst_dataloaders(batch_size=MOIRAI_BATCH_SIZE, device=DEVICE)

## The Fast Evaluation Loop
Since Moirai is frozen, we loop over the patch sizes, instantiate Moirai, and **pre-extract the embeddings ($Z$)**. We then pass these $Z$ loaders to the different head architectures.

In [3]:
# We define a function to run the evaluation for a specific patch size
def evaluate_heads_for_patch(patch):
    
    # 1. Instantiate the Frozen Encoder
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=patch, context_length=36, patch_size=patch, 
        num_samples=100, target_dim=NUM_VARS, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    )
    
    # 2. Pre-Extract Embeddings (Z)
    tr_z, va_z, te_z = get_z_loaders(
        encoder, raw_tr, raw_va, raw_te, 
        head_batch_size=HEAD_BATCH_SIZE, device=DEVICE, remove_last_patch=False, num_vars=NUM_VARS
    )
    
    return tr_z, va_z, te_z

## 1. Mean Pooling Architecture


How it works: The simplest baseline. It takes the contextual patches generated by Moirai and averages them over the temporal dimension. The result is a single flattened vector containing the average representation of each variable, which is then passed to a Linear classifier.

In [4]:
results_mean_pooling = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch) # Extract once
    
    _, metrics = grid_search_heads(
        MeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes},
        tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
    )
    results_mean_pooling[patch] = metrics

In [5]:
df_results_mean_pooling = pd.DataFrame.from_dict(results_mean_pooling, orient='index')
df_results_mean_pooling.index.name = 'Patch Size'
print('results mean pooling')
display(df_results_mean_pooling[['Accuracy']])
df_patch_8_metrics.loc['Mean Pooling'] = df_results_mean_pooling.loc[8]
print('Patch  size = 8')
display(df_patch_8_metrics)

results mean pooling


,Accuracy
Patch Size,
8,0.617194
16,0.618005
32,0.603812
64,0.633009


Patch  size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.617194,0.451334,0.37661,0.387516,0.569912,0.617194,0.5734


## 2. Single-Scale Attention Architecture


How it works: Instead of a naive average, this head uses a learned Query vector. This Query looks at all the patches (Keys/Values) and assigns an attention weight to each. The network learns which patches are the most important for the classification task.
* shared_context: One single Query acts across all variables simultaneously.
* independent_context: Each variable has its own dedicated Query to find its own important patches.

In [6]:
results_attn = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_attn[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleAttentionClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_attn[patch][mode] = metrics
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics

# Accuracy only for all patch sizes
df_attn_acc = pd.DataFrame(
    {patch: {mode: results_attn[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_attn_acc.index.name = 'Mode'
df_attn_acc.columns.name = 'Patch Size'
print('Results Single-Scale Attention - Accuracy')
display(df_attn_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results Single-Scale Attention - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5945,0.5945,0.5908,0.6314
independent_context,0.5880,0.6050,0.5892,0.6237


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.617194,0.451334,0.376610,0.387516,0.569912,0.617194,0.573400
Attention (shared_context),0.594485,0.420022,0.351328,0.359545,0.539370,0.594485,0.547029
Attention (independent_context),0.587997,0.422001,0.341196,0.350519,0.539194,0.587997,0.540742


## 3. Multi-Head Attention (MHA)


How it works: A single attention mechanism might focus entirely on one feature (e.g., the highest peak). Multi-Head Attention solves this by projecting the data into multiple sub-spaces (e.g., 16 heads).
This allows the model to capture multiple distinct temporal concepts simultaneously (e.g., Head 1 looks at recent patches, Head 2 looks at periodic drops, ...).

### 3.1 MHA with 16 heads over all patch sizes

In [7]:
results_mha = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_mha[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 16},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_mha[patch][mode] = metrics

# Accuracy only for all patch sizes
df_mha_acc = pd.DataFrame(
    {patch: {mode: results_mha[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_mha_acc.index.name = 'Mode'
df_mha_acc.columns.name = 'Patch Size'
print('Results MHA (16 heads) - Accuracy')
display(df_mha_acc.astype(float).round(4))

Results MHA (16 heads) - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6103,0.6196,0.5998,0.6168
independent_context,0.6071,0.6212,0.6058,0.6111


### 3.2 MHA over patches of sizes 8 but with highter number of heads

In [8]:
results_mha8 = {}

for heads in HEADS_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(8)
    results_mha8[heads] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_mha8[heads][mode] = metrics
        df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics

In [9]:
# Accuracy by number of heads (all runs are patch=8)
df_mha8_acc = pd.DataFrame(
    {heads: {mode: results_mha8[heads][mode]['Accuracy'] for mode in MODES} for heads in HEADS_TO_TEST}
)
df_mha8_acc.index.name = 'Mode'
df_mha8_acc.columns.name = 'Num Heads'
print('Results MHA (Patch = 8) - Accuracy by number of heads')
display(df_mha8_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results MHA (Patch = 8) - Accuracy by number of heads


Num Heads,8,16,32,64,128
Mode,,,,,
shared_context,0.6010,0.6107,0.6071,0.6034,0.6131
independent_context,0.5884,0.6075,0.6038,0.6071,0.6172


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.617194,0.451334,0.376610,0.387516,0.569912,0.617194,0.573400
Attention (shared_context),0.594485,0.420022,0.351328,0.359545,0.539370,0.594485,0.547029
Attention (independent_context),0.587997,0.422001,0.341196,0.350519,0.539194,0.587997,0.540742
MHA-8 (shared_context),0.600973,0.409083,0.343800,0.347155,0.539841,0.600973,0.547471
MHA-8 (independent_context),0.588402,0.414790,0.339842,0.346094,0.532257,0.588402,0.540413
MHA-16 (shared_context),0.610706,0.456791,0.393829,0.404753,0.562494,0.610706,0.577734
MHA-16 (independent_context),0.607461,0.490473,0.390750,0.404944,0.562726,0.607461,0.574269
MHA-32 (shared_context),0.607056,0.416881,0.361946,0.366258,0.546820,0.607056,0.555537
MHA-32 (independent_context),0.603812,0.416136,0.355354,0.362083,0.544185,0.603812,0.551721
MHA-64 (shared_context),0.603406,0.497533,0.373545,0.382555,0.554082,0.603406,0.548457


## 4. Hierarchical Multi-Head Attention


How it works: Inspired by Hierarchical Attention Networks (HAN). It performs attention in two steps:
1. Temporal Attention: MHA is applied *within* each variable individually to summarize its temporal patches into a single variable-vector.
2. Variable Attention: A second MHA is applied *across* the summarized variables. This allows the network to learn inter-variable correlations.

In [10]:
results_hier = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_hier[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            HierarchicalMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 16},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_hier[patch][mode] = metrics
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical MHA ({mode})"] = metrics

# Accuracy only for all patch sizes
df_hier_acc = pd.DataFrame(
    {patch: {mode: results_hier[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_hier_acc.index.name = 'Mode'
df_hier_acc.columns.name = 'Patch Size'
print('Results Hierarchical MHA (16 heads) - Accuracy')
display(df_hier_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results Hierarchical MHA (16 heads) - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5811,0.5856,0.5803,0.5770
independent_context,0.5839,0.5819,0.5689,0.5823


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.617194,0.451334,0.376610,0.387516,0.569912,0.617194,0.573400
Attention (shared_context),0.594485,0.420022,0.351328,0.359545,0.539370,0.594485,0.547029
Attention (independent_context),0.587997,0.422001,0.341196,0.350519,0.539194,0.587997,0.540742
MHA-8 (shared_context),0.600973,0.409083,0.343800,0.347155,0.539841,0.600973,0.547471
MHA-8 (independent_context),0.588402,0.414790,0.339842,0.346094,0.532257,0.588402,0.540413
MHA-16 (shared_context),0.610706,0.456791,0.393829,0.404753,0.562494,0.610706,0.577734
MHA-16 (independent_context),0.607461,0.490473,0.390750,0.404944,0.562726,0.607461,0.574269
MHA-32 (shared_context),0.607056,0.416881,0.361946,0.366258,0.546820,0.607056,0.555537
MHA-32 (independent_context),0.603812,0.416136,0.355354,0.362083,0.544185,0.603812,0.551721
MHA-64 (shared_context),0.603406,0.497533,0.373545,0.382555,0.554082,0.603406,0.548457


## Final Summary
Review of all architectures evaluated on the frozen Moirai encoder.

In [11]:
print("FINAL SUMMARY — Patch Size = 8 (All Metrics)")
display(df_patch_8_metrics.astype(float).round(4))

print("\nACCURACY ACROSS PATCH SIZES")

print("\nMean Pooling")
display(df_results_mean_pooling[['Accuracy']].astype(float).round(4))

print("\nSingle-Scale Attention")
display(df_attn_acc.astype(float).round(4))

print("\nMHA (16 heads)")
display(df_mha_acc.astype(float).round(4))

print("\nMHA (Patch = 8, varying heads)")
display(df_mha8_acc.astype(float).round(4))

print("\nHierarchical MHA (16 heads)")
display(df_hier_acc.astype(float).round(4))

FINAL SUMMARY — Patch Size = 8 (All Metrics)


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.6172,0.4513,0.3766,0.3875,0.5699,0.6172,0.5734
Attention (shared_context),0.5945,0.4200,0.3513,0.3595,0.5394,0.5945,0.5470
Attention (independent_context),0.5880,0.4220,0.3412,0.3505,0.5392,0.5880,0.5407
MHA-8 (shared_context),0.6010,0.4091,0.3438,0.3472,0.5398,0.6010,0.5475
MHA-8 (independent_context),0.5884,0.4148,0.3398,0.3461,0.5323,0.5884,0.5404
MHA-16 (shared_context),0.6107,0.4568,0.3938,0.4048,0.5625,0.6107,0.5777
MHA-16 (independent_context),0.6075,0.4905,0.3908,0.4049,0.5627,0.6075,0.5743
MHA-32 (shared_context),0.6071,0.4169,0.3619,0.3663,0.5468,0.6071,0.5555
MHA-32 (independent_context),0.6038,0.4161,0.3554,0.3621,0.5442,0.6038,0.5517
MHA-64 (shared_context),0.6034,0.4975,0.3735,0.3826,0.5541,0.6034,0.5485



ACCURACY ACROSS PATCH SIZES

Mean Pooling


,Accuracy
Patch Size,
8,0.6172
16,0.6180
32,0.6038
64,0.6330



Single-Scale Attention


...

Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6103,0.6196,0.5998,0.6168
independent_context,0.6071,0.6212,0.6058,0.6111



MHA (Patch = 8, varying heads)


Num Heads,8,16,32,64,128
Mode,,,,,
shared_context,0.6010,0.6107,0.6071,0.6034,0.6131
independent_context,0.5884,0.6075,0.6038,0.6071,0.6172



Hierarchical MHA (16 heads)


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5811,0.5856,0.5803,0.5770
independent_context,0.5839,0.5819,0.5689,0.5823
